# DataTalks.Club Machine Learning Zoomcamp 2026
## Homework 2: Machine Learning for Regression

- **Student:** Diego Gonzalez (`NativelogXD`)
- **Cohort:** 2026
- **Submission Form:** [https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw02](https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw02)

### Overview
In this homework, we implement **Linear Regression from scratch** using vectorization and linear algebra in NumPy, following the Normal Equation and $L_2$ Ridge Regularization to predict car fuel efficiency (`fuel_efficiency_mpg`).


In [1]:
import numpy as np
import pandas as pd
import urllib.request

# 1. Download or load the pinned 2026 dataset
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(url)

print(f'Total records: {len(df)}')
print(f'Columns: {list(df.columns)}')

# Filter required columns
selected_columns = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
    'fuel_efficiency_mpg'
]
df = df[selected_columns]
df.head()


Total records: 10000
Columns: ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']


---
### Question 1: Column with Missing Values

Find which column in the selected dataset contains missing values.


In [1]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)

missing_col = missing[missing > 0].index[0]
print(f'\nAnswer Question 1: {missing_col}')


Missing values per column:
engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

Answer Question 1: horsepower


---
### Question 2: Median for Variable `horsepower`

What is the median (50% percentile) for the `horsepower` variable?


In [1]:
median_hp = df['horsepower'].median()
print(f'Horsepower 50th percentile (median): {median_hp}')
print(f'Answer Question 2: {int(round(median_hp))}')


Horsepower 50th percentile (median): 254.0
Answer Question 2: 254


---
### Splitting Dataset & Normal Equation Implementation

We split the data into **60% Train**, **20% Validation**, and **20% Test** with deterministic shuffling (`seed=42`).

The Normal Equation with optional Ridge ($L_2$) regularization is:
$$\mathbf{w} = (\mathbf{X}^T\mathbf{X} + r\mathbf{I})^{-1}\mathbf{X}^T\mathbf{y}$$
where the bias term is unregularized ($r_{0,0} = 0$).


In [1]:
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    
    XTX = X.T.dot(X)
    if r > 0.0:
        reg = r * np.eye(XTX.shape[0])
        reg[0, 0] = 0.0  # Do not regularize intercept
        XTX = XTX + reg
        
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

def rmse(y_true, y_pred):
    error = y_pred - y_true
    mse = (error ** 2).mean()
    return np.sqrt(mse)

def split_data(df, seed=42):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    
    idx = np.arange(n)
    np.random.seed(seed)
    np.random.shuffle(idx)
    
    df_shuffled = df.iloc[idx].reset_index(drop=True)
    df_train = df_shuffled.iloc[:n_train].copy()
    df_val = df_shuffled.iloc[n_train:n_train + n_val].copy()
    df_test = df_shuffled.iloc[n_train + n_val:].copy()
    return df_train, df_val, df_test

df_train, df_val, df_test = split_data(df, seed=42)
print(f'Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}')


Train: 6000, Val: 2000, Test: 2000


---
### Question 3: Missing Value Handling (0 vs Mean)

Compare the validation RMSE when filling missing values with `0` vs filling with the `training mean`.


In [1]:
features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

# 1. Imputation with 0
X_train_0 = df_train[features].fillna(0).values
X_val_0 = df_val[features].fillna(0).values
w0_0, w_0 = train_linear_regression_reg(X_train_0, y_train, r=0.0)
rmse_0 = rmse(y_val, w0_0 + X_val_0.dot(w_0))

# 2. Imputation with train mean
hp_mean = df_train['horsepower'].mean()
X_train_mean = df_train[features].fillna({'horsepower': hp_mean}).values
X_val_mean = df_val[features].fillna({'horsepower': hp_mean}).values
w0_m, w_m = train_linear_regression_reg(X_train_mean, y_train, r=0.0)
rmse_mean = rmse(y_val, w0_m + X_val_mean.dot(w_m))

print(f'RMSE (filled with 0):    {rmse_0:.4f} (round: {round(rmse_0, 3)})')
print(f'RMSE (filled with mean): {rmse_mean:.4f} (round: {round(rmse_mean, 3)})')
print(f'Answer Question 3: With mean')


RMSE (filled with 0):    2.2053 (round: 2.205)
RMSE (filled with mean): 2.2018 (round: 2.202)
Answer Question 3: With mean


---
### Question 4: Regularization Parameter $r$

Try different values of $r \in [0, 0.01, 0.1, 1, 5, 10, 100]$ on the 0-imputed data and round to 4 decimals.


In [1]:
r_list = [0, 0.01, 0.1, 1, 5, 10, 100]
print('Regularization Evaluation (4 decimal places):')
for r in r_list:
    w0, w = train_linear_regression_reg(X_train_0, y_train, r=r)
    val_score = rmse(y_val, w0 + X_val_0.dot(w))
    print(f'  r = {r:<5} -> RMSE = {round(val_score, 4):.4f} (raw: {val_score:.6f})')

print('\nAnswer Question 4: 0')


Regularization Evaluation (4 decimal places):
  r = 0     -> RMSE = 2.2053 (raw: 2.205293)
  r = 0.01  -> RMSE = 2.2053 (raw: 2.205293)
  r = 0.1   -> RMSE = 2.2053 (raw: 2.205293)
  r = 1     -> RMSE = 2.2053 (raw: 2.205293)
  r = 5     -> RMSE = 2.2053 (raw: 2.205293)
  r = 10    -> RMSE = 2.2053 (raw: 2.205293)
  r = 100   -> RMSE = 2.2053 (raw: 2.205294)

Answer Question 4: 0


---
### Question 5: Seed Stability Analysis

Evaluate standard deviation of validation RMSE across seeds `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]`.


In [1]:
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
scores = []

for s in seeds:
    df_tr_s, df_va_s, _ = split_data(df, seed=s)
    y_tr_s = df_tr_s['fuel_efficiency_mpg'].values
    y_va_s = df_va_s['fuel_efficiency_mpg'].values
    
    X_tr_s = df_tr_s[features].fillna(0).values
    X_va_s = df_va_s[features].fillna(0).values
    
    w0_s, w_s = train_linear_regression_reg(X_tr_s, y_tr_s, r=0.0)
    score_s = rmse(y_va_s, w0_s + X_va_s.dot(w_s))
    scores.append(score_s)
    print(f'  Seed {s}: RMSE = {score_s:.5f}')

std_dev = np.std(scores)
print(f'\nStandard Deviation of scores: {std_dev:.5f}')
print(f'Answer Question 5: {round(std_dev, 3)}')


  Seed 0: RMSE = 2.23920
  Seed 1: RMSE = 2.20211
  Seed 2: RMSE = 2.16342
  Seed 3: RMSE = 2.20436
  Seed 4: RMSE = 2.20188
  Seed 5: RMSE = 2.24579
  Seed 6: RMSE = 2.26972
  Seed 7: RMSE = 2.19079
  Seed 8: RMSE = 2.21661
  Seed 9: RMSE = 2.20704

Standard Deviation of scores: 0.02878
Answer Question 5: 0.029


---
### Question 6: Test Evaluation with Seed 9 & $r=0.001$

Split with seed 9, combine train + val, fill missing with 0, and evaluate on test.


In [1]:
df_tr_9, df_va_9, df_te_9 = split_data(df, seed=9)
df_full_train = pd.concat([df_tr_9, df_va_9]).reset_index(drop=True)

y_full_train = df_full_train['fuel_efficiency_mpg'].values
y_test = df_te_9['fuel_efficiency_mpg'].values

X_full_train = df_full_train[features].fillna(0).values
X_test = df_te_9[features].fillna(0).values

w0_final, w_final = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)
test_score = rmse(y_test, w0_final + X_test.dot(w_final))

print(f'Model Intercept w0: {w0_final:.4f}')
print(f'Model Coefficients w: {w_final}')
print(f'Test RMSE: {test_score:.5f}')
print(f'Answer Question 6: {round(test_score, 3)}')


Model Intercept w0: -152.8058
Model Coefficients w: [-0.00135373 -0.00044009 -0.00401645  0.10161646]
Test RMSE: 2.23577
Answer Question 6: 2.236


---
### Summary of Form Answers

| Question | Computed Answer | Form Option |
| :--- | :--- | :--- |
| **Question 1** | `horsepower` (877 missing, 8.77%) | `'horsepower'` |
| **Question 2** | `254.0` | `254` |
| **Question 3** | RMSE Mean: `2.202` < RMSE 0: `2.205` | `With mean` |
| **Question 4** | Lowest $r$ achieving min RMSE (`2.2053`) | `0` |
| **Question 5** | Standard deviation = `0.02878` | `0.029` |
| **Question 6** | Test RMSE = `2.23577` | `2.236` |
